# Paper figures

Regenerates the **data figures** used in the paper with a consistent
publication style (white background, 300 DPI) and writes them to `figures/`:

| Output | Paper figure |
|---|---|
| `figures/heatmap_correlation.png` | Correlation across sampled OCI wavelengths |
| `figures/ae_results.png` | Concrete AE: symmetric vs. asymmetric vs. PCA |
| `figures/fsae_results.png` | FSAE: symmetric vs. asymmetric vs. PCA |

Reads pre-computed `score.parquet` results and the sample matrix from S3 — no
model retraining. Requires the S3 credentials/buckets in `.env` (see README).

The architecture diagrams (tensor reshape, symmetric/asymmetric, FSAE) are hand-drawn
and are **not** reproduced here.

In [ ]:
import os
import re

import numpy as np
import pandas as pd
import boto3
import pyarrow.dataset as ds
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()

# ---- Publication style: white background, 300 DPI, tight bbox ----
FIGDIR = "figures"
os.makedirs(FIGDIR, exist_ok=True)

mpl.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.grid": False,
})

bucket = os.getenv("BUCKET_NAME")
results_bucket = os.getenv("RESULTS_BUCKET_NAME")
hyperparameters_bucket = os.getenv("HYPERPARAMETERS_RESULTS_BUCKET_NAME")

## Correlation heatmap

Pearson correlation across the 277 retained wavelengths (UV < 320 nm and the
590–610 nm blue/red detector overlap are excluded), over the first 1000 sampled pixels.

In [ ]:
columns = pd.read_parquet(f"s3://{bucket}/headers.parquet")["0"].values
toscore = ds.dataset(f"s3://{bucket}/samples/", format="parquet").to_table().to_pandas()
toscore.columns = columns

# Exclude UV < 320 nm and the 590-610 nm blue/red detector overlap -> 277 wavelengths
bad_wavelengths = [c for c in toscore.columns if c < 320 or (c > 590 and c < 610)]
X = toscore.drop(columns=bad_wavelengths).dropna()
print("X shape:", X.shape)

In [ ]:
corr = X.iloc[:1000].corr()

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(corr, cmap="rocket", ax=ax, cbar_kws={"label": "Pearson correlation"})

ax.set_title("Correlation across sampled OCI wavelengths")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Wavelength (nm)")

# Sparse, readable wavelength ticks
wl = [int(c) for c in corr.columns]
step = max(1, len(wl) // 12)
ticks = np.arange(0, len(wl), step)
ax.set_xticks(ticks)
ax.set_xticklabels([wl[i] for i in ticks], rotation=45, ha="right")
ax.set_yticks(ticks)
ax.set_yticklabels([wl[i] for i in ticks], rotation=0)

fig.savefig(os.path.join(FIGDIR, "heatmap_correlation.png"))
plt.show()

## Load result scores

`score.parquet` files are read directly from S3. The PCA baseline comes from the
framework-results bucket; the CAE/FSAE layer sweeps come from the hyperparameter bucket.
Only the latest version per model is used.

In [ ]:
import io

s3 = boto3.client("s3")

def read_parquet_s3(bkt, key):
    """Fetch a parquet object via boto3 and read from memory.

    More robust than pyarrow's native s3:// filesystem, which can stall on
    cross-region redirects for these result buckets.
    """
    obj = s3.get_object(Bucket=bkt, Key=key)
    return pd.read_parquet(io.BytesIO(obj["Body"].read()))

def latest_scores(bkt, pattern, index_fn):
    """Read the latest-version score.parquet per model from a bucket."""
    paths = {}
    for obj in s3.list_objects_v2(Bucket=bkt).get("Contents", []):
        m = re.match(pattern, obj["Key"])
        if not m:
            continue
        *key_parts, version = m.groups()
        version = int(version)
        mk = "/".join(key_parts)
        if mk not in paths or version > paths[mk][0]:
            paths[mk] = (version, obj["Key"])
    dfs = []
    for mk, (version, key) in paths.items():
        df = read_parquet_s3(bkt, key).T  # read from the same bucket we listed
        df.index = [index_fn(mk, version)]
        dfs.append(df)
    return pd.concat(dfs)

# PCA baseline (framework results)
combined_scores = latest_scores(
    results_bucket,
    r"([^/]+)/v(\d+)/score\.parquet$",
    lambda mk, v: f"{mk}_v{v}",
)
pca_baseline = combined_scores.loc["pca_v1", "test_loss"]

# CAE / FSAE layer sweeps (hyperparameter results)
hyperparameter_scores = latest_scores(
    hyperparameters_bucket,
    r"([^/]+)/([^/]+)/(\d+)/v(\d+)/score\.parquet$",
    lambda mk, v: f"{mk.replace('concrete-auto-encoder', 'cae').replace('_architecture', '')}_v{v}",
)
print(f"PCA baseline test loss: {pca_baseline:.6f}")

## Concrete autoencoder (CAE) results

In [ ]:
cae_asym = hyperparameter_scores[hyperparameter_scores.index.str.contains("cae/asymmetric")].sort_values("layers")
cae_sym = hyperparameter_scores[hyperparameter_scores.index.str.contains("cae/symmetric")].sort_values("layers")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(cae_asym["layers"], cae_asym["test_loss"], marker="o", linewidth=1.8, label="Asymmetric")
ax.plot(cae_sym["layers"], cae_sym["test_loss"], marker="s", linestyle="--", linewidth=1.8, label="Symmetric")
ax.axhline(pca_baseline, color="0.4", linestyle=":", linewidth=1.5, label="PCA baseline")

ax.set_title("Concrete autoencoder: symmetric vs. asymmetric")
ax.set_xlabel("Encoded dimension")  # the 'layers' column holds the encoder size (16-64)
ax.set_ylabel("Test loss")
ax.grid(True, axis="y", linestyle="--", linewidth=0.5, alpha=0.6)
ax.legend(frameon=False)

fig.savefig(os.path.join(FIGDIR, "ae_results.png"))
plt.show()

## Feature selection autoencoder (FSAE) results

In [ ]:
fsae_asym = hyperparameter_scores[hyperparameter_scores.index.str.contains("fsae/asymmetric")].sort_values("layers")
fsae_sym = hyperparameter_scores[hyperparameter_scores.index.str.contains("fsae/symmetric")].sort_values("layers")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(fsae_asym["layers"], fsae_asym["test_loss"], marker="o", linewidth=1.8, label="Asymmetric")
ax.plot(fsae_sym["layers"], fsae_sym["test_loss"], marker="s", linestyle="--", linewidth=1.8, label="Symmetric")
ax.axhline(pca_baseline, color="0.4", linestyle=":", linewidth=1.5, label="PCA baseline")

ax.set_title("Feature selection autoencoder: symmetric vs. asymmetric")
ax.set_xlabel("Encoded dimension")  # the 'layers' column holds the encoder size (16-64)
ax.set_ylabel("Test loss")
ax.grid(True, axis="y", linestyle="--", linewidth=0.5, alpha=0.6)
ax.legend(frameon=False)

fig.savefig(os.path.join(FIGDIR, "fsae_results.png"))
plt.show()